In [ ]:
import pandas as pd

In [ ]:
# # weather columns
# X = ['temperature_2m_c',
#     'wind_speed_100m_ms',
#     'wind_gusts_10m_ms',
#     'cloud_cover_pct',
#     'shortwave_radiation_wm2',
#     'direct_radiation_wm2',
#     'diffuse_radiation_wm2',
#     'pressure_msl_hpa',
#     'snowfall_cm',
#     'rain_mm',
#     'precipitation_mm'
#     ]

In [ ]:
# include lag features

# y = ['Biomass',
#     'Fossil Gas',
#     'Fossil Hard coal',
#     'Fossil Oil',
#     'Hydro Pumped Storage',
#     'Hydro Run-of-river and poundage',
#     'Nuclear',
#     'Other',
#     'Solar',
#     'Wind Offshore',
#     'Wind Onshore',
#     'TotalOutput-MW'
#     ]

In [ ]:
# lag_features = ['Solar','Wind Onshore','Wind Offshore','total_output_MW']

# LSTM pipeline

Below is a clean end-to-end Python pipeline for your case:

 - handles 30-minute time series
 - imputes missing values correctly
 - adds time features
 - adds lagged generation
 - creates LSTM sequences
 - trains the model

In [23]:
# Raw data
#    ↓
# Time interpolation
#    ↓
# Feature engineering
#    ↓
# Lagged generation
#    ↓
# Scaling
#    ↓
# Sequence creation
#    ↓
# LSTM
#    ↓
# Fuel type generation prediction

In [4]:
from google.cloud import bigquery

PROJECT = "gridzero-489711"
DATASET = "merged_set"
TABLE = "test_merge_2017_onward_raw"

query = f"""
    SELECT *
    FROM {PROJECT}.{DATASET}.{TABLE}
"""

client = bigquery.Client('gridzero-489711')
query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()

/Users/jamesla/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [5]:
df.head()

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity_gCO2_kWh,status
0,2017-09-12 00:00:00,11.6,31.0,28.1,4,0.0,0.0,0.0,1001.2,0.0,...,0.0,265.0,7897.0,173.0,0.0,4006.573,4394.973,21722.546,142.0,okay
1,2017-09-12 00:30:00,11.6,31.0,28.1,4,0.0,0.0,0.0,1001.2,0.0,...,0.0,248.0,7897.0,174.0,0.0,3973.985,4418.446,21554.431,140.0,okay
2,2017-09-12 01:00:00,11.2,30.3,27.0,5,0.0,0.0,0.0,1001.9,0.0,...,0.0,242.0,7852.0,171.0,0.0,3941.698,4533.019,21767.717,139.0,okay
3,2017-09-12 01:30:00,11.2,30.3,27.0,5,0.0,0.0,0.0,1001.9,0.0,...,0.0,242.0,7706.0,174.0,0.0,3945.620,4726.191,21742.811,137.0,okay
4,2017-09-12 02:00:00,10.9,29.6,25.2,7,0.0,0.0,0.0,1002.4,0.0,...,0.0,242.0,7684.0,173.0,0.0,3919.454,4832.410,21579.864,132.0,okay


In [9]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [10]:
df["datetime"] = pd.to_datetime(df["datetime"])

df = df.sort_values("datetime")
df = df.set_index("datetime")

In [11]:
# Handle Missing Data

# interpolate using time
df = df.interpolate(method="time")

# fill remaining edges
df = df.ffill().bfill()

/var/folders/0s/rkp2dnc532q3ycyt389hy7f00000gn/T/ipykernel_12615/601437718.py:2: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.interpolate(method="time")


In [12]:
df.isna().sum()

temperature_2m_c                   0
wind_speed_100m_ms                 0
wind_gusts_10m_ms                  0
cloud_cover_pct                    0
shortwave_radiation_wm2            0
direct_radiation_wm2               0
diffuse_radiation_wm2              0
pressure_msl_hpa                   0
snowfall_cm                        0
rain_mm                            0
precipitation_mm                   0
Biomass                            0
Fossil Gas                         0
Fossil Hard coal                   0
Fossil Oil                         0
Hydro Pumped Storage               0
Hydro Run-of-river and poundage    0
Nuclear                            0
Other                              0
Solar                              0
Wind Offshore                      0
Wind Onshore                       0
TotalOutput-MW                     0
carbon_intensity_gCO2_kWh          0
status                             0
dtype: int64

In [15]:
# Add Time Features

df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

In [16]:
# cyclical encoding
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

df["dow_sin"] = np.sin(2*np.pi*df["dayofweek"]/7)
df["dow_cos"] = np.cos(2*np.pi*df["dayofweek"]/7)

In [19]:
# Lagged Generation Features

lag_features = [
    "Solar",
    "Wind Onshore",
    "Wind Offshore",
    "TotalOutput-MW"
]

for col in lag_features:
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_lag48"] = df[col].shift(48) 

In [21]:
df = df.dropna()

In [22]:
df

,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,rain_mm,...,dow_sin,dow_cos,Solar_lag1,Solar_lag48,Wind Onshore_lag1,Wind Onshore_lag48,Wind Offshore_lag1,Wind Offshore_lag48,TotalOutput-MW_lag1,TotalOutput-MW_lag48
datetime,,,,,,,,,,,,,,,,,,,,,
2017-09-13 00:00:00,15.6,58.0,61.6,21.0,0.0,0.0,0.0,993.0,0.0,0.5,...,0.974928,-0.222521,0.0,0.0,2439.273,4394.973,3880.903,4006.573,20596.176,21722.546
2017-09-13 00:30:00,15.6,58.0,61.6,21.0,0.0,0.0,0.0,993.0,0.0,0.5,...,0.974928,-0.222521,0.0,0.0,2465.004,4418.446,3941.074,3973.985,20278.078,21554.431
2017-09-13 01:00:00,15.4,55.2,60.8,74.0,0.0,0.0,0.0,992.9,0.0,0.0,...,0.974928,-0.222521,0.0,0.0,2501.234,4533.019,4019.167,3941.698,20200.401,21767.717
2017-09-13 01:30:00,15.4,55.2,60.8,74.0,0.0,0.0,0.0,992.9,0.0,0.0,...,0.974928,-0.222521,0.0,0.0,2623.010,4726.191,4013.885,3945.620,20494.895,21742.811
2017-09-13 02:00:00,14.9,60.4,63.7,48.0,0.0,0.0,0.0,992.8,0.0,0.0,...,0.974928,-0.222521,0.0,0.0,2753.792,4832.410,4035.308,3919.454,20233.100,21579.864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-12 21:00:00,11.4,50.8,68.0,100.0,0.0,0.0,0.0,1004.1,0.0,0.0,...,0.433884,-0.900969,2492.0,0.0,9695.758,9032.398,11622.834,10276.025,34057.592,32453.423
2026-03-12 21:30:00,11.4,50.8,68.0,100.0,0.0,0.0,0.0,1004.1,0.0,0.0,...,0.433884,-0.900969,2492.0,0.0,9695.758,8973.533,11622.834,10034.962,34057.592,31433.495
2026-03-12 22:00:00,11.4,51.5,67.3,100.0,0.0,0.0,0.0,1003.0,0.0,0.1,...,0.433884,-0.900969,2492.0,0.0,9695.758,8867.501,11622.834,10001.928,34057.592,31148.429


# features and Targets

In [24]:
weather_cols = [
'temperature_2m_c',
'wind_speed_100m_ms',
'wind_gusts_10m_ms',
'cloud_cover_pct',
'shortwave_radiation_wm2',
'direct_radiation_wm2',
'diffuse_radiation_wm2',
'pressure_msl_hpa',
'snowfall_cm',
'rain_mm',
'precipitation_mm'
]

In [25]:
time_cols = [
"hour_sin","hour_cos",
"dow_sin","dow_cos"
]

In [26]:
lag_cols = [
"Solar_lag1","Solar_lag48",
"Wind Onshore_lag1","Wind Onshore_lag48",
"Wind Offshore_lag1","Wind Offshore_lag48",
"total_output_MW_lag1","total_output_MW_lag48"
]